In [1]:
pip install stable-baselines3 gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 4.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import random
import math
import gymnasium as gym
from gymnasium import spaces


# ================= CONFIG =================
WIDTH, HEIGHT = 640, 760
RADIUS = 20
DIAMETER = 40

ROWS, COLS = 10, 9
GRID_WIDTH = COLS * DIAMETER
OFFSET_X = (WIDTH - GRID_WIDTH) // 2

SHOOT_Y = HEIGHT - 90
BOTTOM_LIMIT = HEIGHT - 120

MAX_SHOTS = 50

GATES = ["H", "CNOT", "T", "TD", "S", "SD"]

gate_to_id = {g: i + 1 for i, g in enumerate(GATES)}
id_to_gate = {i + 1: g for i, g in enumerate(GATES)}

ANGLES = np.linspace(10, 170, 17)


# ================= ENV =================
class QuantumBubbleShooterEnv(gym.Env):

    def __init__(self):
        super().__init__()

        self.action_space = spaces.Discrete(len(ANGLES))

        self.observation_space = spaces.Box(
            low=0,
            high=50,
            shape=(ROWS * COLS + 3,),
            dtype=np.float32
        )

        self.reset()

    # ================= RESET =================
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.grid = [[None for _ in range(COLS)] for _ in range(ROWS)]

        for r in range(3):
            for c in range(COLS):
                if random.random() < 0.8:
                    self.grid[r][c] = random.choice(GATES)

        self.current_gate = random.choice(GATES)
        self.next_gate = random.choice(GATES)

        self.shots = 0
        self.score = 0

        return self._get_obs(), {}

    # ================= OBS =================
    def _get_obs(self):

        flat_grid = []

        for row in self.grid:
            for cell in row:
                flat_grid.append(
                    0 if cell is None else gate_to_id[cell]
                )

        return np.array(
            flat_grid +
            [
                gate_to_id[self.current_gate],
                gate_to_id[self.next_gate],
                MAX_SHOTS - self.shots
            ],
            dtype=np.float32
        )

    # ================= HELPERS =================
    def grid_to_pixel(self, r, c):
        return (
            OFFSET_X + c * DIAMETER + RADIUS,
            r * DIAMETER + RADIUS + 80
        )

    def nearest_cell(self, x, y):

        best = None
        best_dist = 999999

        for r in range(ROWS):
            for c in range(COLS):

                if self.grid[r][c] is None:

                    gx, gy = self.grid_to_pixel(r, c)

                    d = math.hypot(x - gx, y - gy)

                    if d < best_dist:
                        best_dist = d
                        best = (r, c)

        return best

    def neighbors(self, r, c):

        dirs = [(1,0),(-1,0),(0,1),(0,-1)]

        result = []

        for dr, dc in dirs:

            nr = r + dr
            nc = c + dc

            if 0 <= nr < ROWS and 0 <= nc < COLS:
                result.append((nr, nc))

        return result

    # ================= QUANTUM RULES =================
    def can_pop(self, a, b):

        if a == b and a in ["H", "CNOT"]:
            return True

        if (a == "T" and b == "TD") or (a == "TD" and b == "T"):
            return True

        if (a == "S" and b == "SD") or (a == "SD" and b == "S"):
            return True

        return False

    # ================= RESOLVE =================
    def resolve(self, r, c):

        reward = -1

        base = self.grid[r][c]

        for nr, nc in self.neighbors(r, c):

            if self.grid[nr][nc]:

                if self.can_pop(base, self.grid[nr][nc]):

                    self.grid[r][c] = None
                    self.grid[nr][nc] = None

                    reward = 10
                    self.score += 10

                    break

        return reward

    # ================= SIMULATE SHOT =================
    def simulate_shot(self, angle):

        rad = math.radians(angle)

        x = WIDTH // 2
        y = SHOOT_Y

        vx = 7 * math.cos(rad)
        vy = -7 * math.sin(rad)

        while True:

            x += vx
            y += vy

            # Bounce walls
            if x <= OFFSET_X + RADIUS or x >= OFFSET_X + GRID_WIDTH - RADIUS:
                vx *= -1

            # Hit ceiling
            if y <= 80:
                break

            # Hit bubble collision
            collision = False

            for r in range(ROWS):
                for c in range(COLS):

                    if self.grid[r][c]:

                        gx, gy = self.grid_to_pixel(r, c)

                        if math.hypot(x - gx, y - gy) <= DIAMETER - 2:
                            collision = True
                            break

                if collision:
                    break

            if collision:
                break

        return self.nearest_cell(x, y)

    # ================= STEP =================
    def step(self, action):

        self.shots += 1

        terminated = False
        truncated = False

        angle = ANGLES[action]

        placement = self.simulate_shot(angle)

        reward = -1

        if placement:

            r, c = placement

            self.grid[r][c] = self.current_gate

            reward = self.resolve(r, c)

        else:
            reward = -10

        # Update gate queue
        self.current_gate = self.next_gate
        self.next_gate = random.choice(GATES)

        # Win condition
        if all(
            self.grid[r][c] is None
            for r in range(ROWS)
            for c in range(COLS)
        ):
            reward += 100
            terminated = True

        # Lose by shots
        if self.shots >= MAX_SHOTS:
            reward -= 50
            terminated = True

        # Lose by overflow
        for r in range(ROWS):
            for c in range(COLS):

                if self.grid[r][c]:

                    _, y = self.grid_to_pixel(r, c)

                    if y >= BOTTOM_LIMIT:
                        reward -= 100
                        terminated = True

        return self._get_obs(), reward, terminated, truncated, {}

In [3]:
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

# =============================
# CREATE ENV
# =============================
env = QuantumBubbleShooterEnv()

# (optional but recommended) check env validity
check_env(env, warn=True)

# =============================
# PPO MODEL (IMPROVED SETUP)
# =============================
model = PPO(
    policy="MlpPolicy",
    env=env,
    verbose=1,

    # --- CORE IMPROVEMENTS ---
    learning_rate=3e-4,
    n_steps=2048,        # IMPORTANT: more stable learning than 512
    batch_size=64,
    n_epochs=10,
    gamma=0.99,

    # --- EXPLORATION BOOST ---
    ent_coef=0.01,       # helps prevent early convergence to bad policy

    # --- STABILITY ---
    clip_range=0.2,
)

# =============================
# TRAIN
# =============================
model.learn(total_timesteps=100000)

# =============================
# SAVE MODEL
# =============================
model.save("quantum_bubble_agent_v1")

print("Training complete. Model saved.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | 15.5     |
| time/              |          |
|    fps             | 283      |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 2048     |
---------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | 12.9        |
| time/                   |             |
|    fps                  | 269         |
|    iterations           | 2           |
|    time_elapsed         | 15          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010889252 |
|    clip_fraction        | 0.0864      |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.83       |
|    explained_variance   | 0.00525     |
|    learning_rate        | 0.0003      |
|    loss                 | 92.8        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.025      |
|    value_loss           | 304         |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50    

In [4]:
obs, _ = env.reset()

for i in range(50):
    action, _ = model.predict(obs)

    obs, reward, done, _, _ = env.step(action)

    print(action, reward)

    if done:
        break

4 -1
5 -1
16 -1
7 -1
7 -1
13 10
3 10
4 10
8 -1
7 -1
3 -1
8 -1
0 -1
8 10
9 -1
16 -1
16 -1
3 -1
7 -1
16 -1
13 -1
1 -1
7 -1
14 10
2 -1
13 -1
11 -1
8 -1
16 -1
9 -1
16 -1
14 10
7 -1
7 -1
8 -1
14 10
8 -1
15 -1
16 10
7 -1
7 -1
8 -1
13 -1
14 -1
0 -1
1 -1
16 -1
2 -1
7 -1
7 -40


In [5]:
import pygame, math, random, imageio, os
import numpy as np

# HEADLESS MODE FOR COLAB
os.environ["SDL_VIDEODRIVER"] = "dummy"

pygame.init()

# ---------------- CONFIG ----------------
WIDTH, HEIGHT = 640, 760
RADIUS = 20
DIAMETER = 40

ROWS, COLS = 10, 9
GRID_WIDTH = COLS * DIAMETER
OFFSET_X = (WIDTH - GRID_WIDTH) // 2

SHOOT_Y = HEIGHT - 90
BOTTOM_LIMIT = HEIGHT - 120
MAX_SHOTS = 20

GATES = ["H", "CNOT", "T", "TD", "S", "SD"]

COLORS = {
    "H": (255,100,100),
    "CNOT": (100,255,150),
    "T": (100,100,255),
    "TD": (150,150,255),
    "S": (255,220,100),
    "SD": (230,200,120)
}

screen = pygame.Surface((WIDTH, HEIGHT))

font = pygame.font.SysFont("Arial", 16)

# ---------------- GAME STATE ----------------
grid = [[None for _ in range(COLS)] for _ in range(ROWS)]

for r in range(3):
    for c in range(COLS):
        if random.random() < 0.8:
            grid[r][c] = random.choice(GATES)

current_gate = random.choice(GATES)
next_gate = random.choice(GATES)

score = 0
shots = 0

bubble_pos = None
bubble_vel = None

frames = []

# ---------------- HELPERS ----------------
def grid_to_pixel(r,c):
    return OFFSET_X + c*DIAMETER + RADIUS, r*DIAMETER + RADIUS + 80


def nearest_cell(x,y):
    best=None
    best_dist=9999

    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] is None:
                gx,gy = grid_to_pixel(r,c)

                d = math.hypot(x-gx,y-gy)

                if d < best_dist:
                    best_dist=d
                    best=(r,c)

    return best


def draw_gate(g,x,y):

    pygame.draw.circle(screen,COLORS[g],(x,y),RADIUS)

    txt = font.render(g,True,(255,255,255))
    rect = txt.get_rect(center=(x,y))

    screen.blit(txt,rect)


def draw_ui():

    screen.fill((20,20,30))

    for r in range(ROWS):
        for c in range(COLS):

            if grid[r][c]:

                x,y = grid_to_pixel(r,c)

                draw_gate(grid[r][c],x,y)

    pygame.draw.circle(screen,(80,80,100),(WIDTH//2,SHOOT_Y),25)

    draw_gate(current_gate, WIDTH//2, SHOOT_Y)

    score_txt = font.render(f"Score:{score}",True,(255,255,255))
    screen.blit(score_txt,(20,20))


def can_pop(a,b):

    if a==b and a in ["H","CNOT"]:
        return True

    if (a=="T" and b=="TD") or (a=="TD" and b=="T"):
        return True

    if (a=="S" and b=="SD") or (a=="SD" and b=="S"):
        return True

    return False


def neighbors(r,c):

    dirs=[(1,0),(-1,0),(0,1),(0,-1)]

    result=[]

    for dr,dc in dirs:

        nr=r+dr
        nc=c+dc

        if 0<=nr<ROWS and 0<=nc<COLS:
            result.append((nr,nc))

    return result


def resolve(r,c):
    global score

    base = grid[r][c]

    for nr,nc in neighbors(r,c):

        if grid[nr][nc]:

            if can_pop(base,grid[nr][nc]):

                grid[r][c]=None
                grid[nr][nc]=None

                score += 10
                return


# ---------------- SIMULATION LOOP ----------------
while shots < MAX_SHOTS:

    angle = random.randint(10,170)

    rad = math.radians(angle)

    bubble_pos=[WIDTH//2,SHOOT_Y]
    bubble_vel=[7*math.cos(rad),-7*math.sin(rad)]

    shots += 1

    while bubble_pos:

        bubble_pos[0]+=bubble_vel[0]
        bubble_pos[1]+=bubble_vel[1]

        if bubble_pos[0]<=OFFSET_X+RADIUS or bubble_pos[0]>=OFFSET_X+GRID_WIDTH-RADIUS:
            bubble_vel[0]*=-1

        hit=False

        for r in range(ROWS):
            for c in range(COLS):

                if grid[r][c]:

                    gx,gy = grid_to_pixel(r,c)

                    if math.hypot(bubble_pos[0]-gx,bubble_pos[1]-gy)<=DIAMETER-2:
                        hit=True
                        break

            if hit:
                break

        if bubble_pos[1]<=80 or hit:

            r,c = nearest_cell(bubble_pos[0],bubble_pos[1])

            grid[r][c]=current_gate

            resolve(r,c)

            bubble_pos=None

            current_gate=next_gate
            next_gate=random.choice(GATES)

        draw_ui()

        if bubble_pos:
            draw_gate(current_gate,int(bubble_pos[0]),int(bubble_pos[1]))

        frame = pygame.surfarray.array3d(screen)
        frame = np.transpose(frame,(1,0,2))

        frames.append(frame)

# ---------------- SAVE GIF ----------------
imageio.mimsave("quantum_bubble.gif", frames, fps=20)

print("GIF SAVED!")



/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

GIF SAVED!
